# I. 🔍 AUTO CHURN DETECTION
1. 📌 Logic

Churn = không quay lại sau X ngày

👉 Flexible theo dataset:

Nếu có customer_id + date → dùng behavioral churn
Nếu có sẵn churn column → dùng luôn

2. 🧠 Code Auto Detect Churn

In [ ]:
def detect_churn(df, customer_col, date_col, threshold_days=30):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # last activity
    last_date = df[date_col].max()

    last_activity = df.groupby(customer_col)[date_col].max().reset_index()
    last_activity["days_inactive"] = (last_date - last_activity[date_col]).dt.days

    # churn label
    last_activity["churn"] = (last_activity["days_inactive"] > threshold_days).astype(int)

    return last_activity

churn_df = detect_churn(df, "customer_id", "date")
churn_df.head()

3. 📊 Churn Rate

In [ ]:
def churn_rate(churn_df):
    rate = churn_df["churn"].mean()
    print(f"Churn Rate: {rate:.2%}")

churn_rate(churn_df)

👉 Insight:

“30% user đã churn → vấn đề retention”

4. 🔥 Churn by Segment

In [ ]:
def churn_by_segment(df, churn_df, customer_col, segment_col):
    merged = df[[customer_col, segment_col]].drop_duplicates().merge(churn_df, on=customer_col)

    result = merged.groupby(segment_col)["churn"].mean().sort_values(ascending=False)

    result.plot(kind="bar", title=f"Churn by {segment_col}")
    plt.show()

churn_by_segment(df, churn_df, "customer_id", "device")

👉 Insight:

“Mobile churn cao hơn → UX issue”

5. ⏳ Retention Curve

In [ ]:
def retention_curve(df, customer_col, date_col):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    first = df.groupby(customer_col)[date_col].min()
    df["cohort"] = df[customer_col].map(first)

    df["days_since"] = (df[date_col] - df["cohort"]).dt.days

    retention = df.groupby("days_since")[customer_col].nunique()

    retention = retention / retention.iloc[0]

    retention.plot(title="Retention Curve")
    plt.show()

retention_curve(df, "customer_id", "date")

👉 Insight:

“Drop mạnh ở ngày 7 → onboarding issue”
# II. 🔻 AUTO FUNNEL DETECTION
1. 📌 Requirement

Dataset dạng:

- user_id
- event (view, add_to_cart, checkout…)
- timestamp

2. 🧠 Build Funnel

In [ ]:
def build_funnel(df, user_col, event_col, funnel_steps):
    funnel_counts = []

    for step in funnel_steps:
        users = df[df[event_col] == step][user_col].nunique()
        funnel_counts.append(users)

    funnel_df = pd.DataFrame({
        "step": funnel_steps,
        "users": funnel_counts
    })

    funnel_df["conversion_rate"] = funnel_df["users"] / funnel_df["users"].shift(1)
    funnel_df.loc[0, "conversion_rate"] = 1

    return funnel_df

funnel_steps = ["visit", "add_to_cart", "checkout", "purchase"]

funnel_df = build_funnel(df, "customer_id", "event", funnel_steps)
funnel_df

3. 📊 Funnel Visualization

In [ ]:
def plot_funnel(funnel_df):
    sns.barplot(x="step", y="users", data=funnel_df)
    plt.title("Funnel Users")
    plt.show()

plot_funnel(funnel_df)

4. 🚨 Drop-off Detection (🔥 quan trọng)

In [ ]:
def detect_drop_off(funnel_df):
    funnel_df["drop_off"] = 1 - funnel_df["conversion_rate"]

    worst_step = funnel_df.sort_values("drop_off", ascending=False).iloc[0]

    print("Biggest drop at:", worst_step["step"])
    print(f"Drop rate: {worst_step['drop_off']:.2%}")

detect_drop_off(funnel_df)

👉 Insight:

“Checkout drop 70% → payment issue”
# III. 🎯 AUTO INSIGHT (CHURN + FUNNEL)

In [ ]:
def churn_funnel_insight(churn_df, funnel_df):
    insights = []

    churn_rate = churn_df["churn"].mean()
    if churn_rate > 0.3:
        insights.append("High churn rate detected")

    max_drop = funnel_df["drop_off"].max()
    if max_drop > 0.5:
        insights.append("Severe funnel drop detected")

    return insights

print(churn_funnel_insight(churn_df, funnel_df))

# IV. 🧠 STORYTELLING OUTPUT

👉 Ví dụ auto output:

What

Churn rate = 35%
70% user drop ở checkout

So what

Mất revenue lớn do user không hoàn tất purchase

Now what

Optimize checkout UX + payment flow

⚠️ GÓC THẲNG

Sai lầm phổ biến:

- Chỉ tính churn rate → không actionable
- Funnel chỉ vẽ → không tìm drop lớn nhất
- Không segment